In [ ]:
%load_ext autoreload
%autoreload 2

# core.dom.summary.object


>DOM summary 
output-file: core.dom.summary.object
title: core.dom.summary.object

This notebook provides utility functions for DOM summary
---

In [ ]:
# | default_exp core.dom.summary.object

In [ ]:
# | export
from IPython.core.interactiveshell import InteractiveShell
from nbdev.showdoc import *

shell = InteractiveShell.instance()
shell.ast_node_interactivity = "all"

In [ ]:
#| export
from fastcore.test import *


In [ ]:
# | export
import asyncio
from contextvars import ContextVar
from functools import wraps
from inspect import signature
import json
import os
import re
import sys
from collections.abc import Callable
from pathlib import Path
from typing import Optional
from weakref import WeakKeyDictionary


project_root = next(
    (path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (path / "pyproject.toml").exists()),
    None,
 )
if project_root is not None and str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from ollama import AsyncClient
try:
    from openai import AsyncOpenAI
except ModuleNotFoundError:
    class AsyncOpenAI:  # type: ignore[no-redef]
        pass

from ribosome.core.dom.utils import (
    get_summary_response_async, 
    get_image_summary_response_async, 
    get_text_summary_response_async,
    image_link_pattern,
    get_image_summary_async,
 )

from ribosome.core.dom.utils import with_async_context


class _LocatedSummaryError(ValueError):
    """An error already carrying the originating AST location."""


def _positive_setting(value, name):
    try:
        if isinstance(value, bool):
            raise ValueError
        parsed = int(value)
        if str(parsed) != str(value).strip() or parsed < 1:
            raise ValueError
    except (ValueError, TypeError):
        raise ValueError(f"{name} must be a positive integer; got {value!r}") from None
    return parsed


_SUMMARY_CONCURRENCY = _positive_setting(os.getenv('RIBOSOME_SUMMARY_CONCURRENCY', '8'), 'RIBOSOME_SUMMARY_CONCURRENCY')
_SUMMARY_TASK_LIMIT = _positive_setting(os.getenv('RIBOSOME_SUMMARY_TASK_LIMIT', '8'), 'RIBOSOME_SUMMARY_TASK_LIMIT')
_summary_semaphores = WeakKeyDictionary()
_summary_budget = ContextVar('summary_budget', default=None)


def _get_summary_semaphore():
    loop = asyncio.get_running_loop()
    limit = _positive_setting(_SUMMARY_CONCURRENCY, 'RIBOSOME_SUMMARY_CONCURRENCY')
    semaphore = _summary_semaphores.get(loop)
    if semaphore is None:
        semaphore = _summary_semaphores[loop] = asyncio.Semaphore(limit)
    return semaphore


def _with_summary_context(message):
    """Bind actual arguments before locating failures; cancellation passes through."""
    def decorate(fn):
        sig = signature(fn)

        @wraps(fn)
        async def wrapped(*args, **kwargs):
            bound = sig.bind(*args, **kwargs)
            bound.apply_defaults()
            token = None
            if _summary_budget.get() is None:
                limit = _positive_setting(_SUMMARY_TASK_LIMIT, 'RIBOSOME_SUMMARY_TASK_LIMIT')
                _positive_setting(_SUMMARY_CONCURRENCY, 'RIBOSOME_SUMMARY_CONCURRENCY')
                token = _summary_budget.set({'limit': limit, 'active': 0})
            try:
                return await fn(*args, **kwargs)
            except _LocatedSummaryError:
                raise
            except Exception as exc:
                raise _node_error(f'{message}: {exc}', bound.arguments.get('path', '$'), bound.arguments['node']) from exc
            finally:
                if token is not None:
                    _summary_budget.reset(token)
        return wrapped
    return decorate


async def _ordered_summary_map(fn, values):
    """At most task_limit spawned workers per traversal, including nested maps.

    Single-item maps run inline without reserving a slot. Reserved slots stay
    with workers until completion. Nested maps run inline when
    all slots are occupied; they never wait for parents to release a task slot.
    Result storage is O(n); active tasks are bounded independently of AST width.
    """
    budget = _summary_budget.get()
    token = None
    if budget is None:
        limit = _positive_setting(_SUMMARY_TASK_LIMIT, 'RIBOSOME_SUMMARY_TASK_LIMIT')
        budget = {'limit': limit, 'active': 0}
        token = _summary_budget.set(budget)
    tasks = []
    try:
        if len(values) == 1:
            return [await fn(0, values[0])]
        results = [None] * len(values)
        count = min(len(values), budget['limit'] - budget['active'])
        if count == 0:
            for index, value in enumerate(values):
                results[index] = await fn(index, value)
            return results
        pending = iter(enumerate(values))

        async def worker():
            try:
                for index, value in pending:
                    results[index] = await fn(index, value)
            finally:
                budget['active'] -= 1

        for _ in range(count):
            # Reservation is synchronous before spawning, shared by descendants.
            budget['active'] += 1
            tasks.append(asyncio.create_task(worker()))
        try:
            await asyncio.gather(*tasks)
        except BaseException:
            for task in tasks:
                task.cancel()
            await asyncio.gather(*tasks, return_exceptions=True)
            raise
        return results
    finally:
        if token is not None:
            _summary_budget.reset(token)


def _summary_child_path(path, key):
    if isinstance(key, int):
        return f'{path}[{key}]'
    if re.fullmatch(r'[A-Za-z_][A-Za-z_0-9]*', key):
        return f'{path}.{key}'
    return f'{path}[{json.dumps(key, ensure_ascii=False)}]'


def _apply_summary_action(node, action, path):
    """Once per traversed value; replacements preserve dict/list/scalar category."""
    transformed = action(node) if action is not None else node
    for category in (dict, list):
        if isinstance(node, category) != isinstance(transformed, category):
            raise _node_error('Action must preserve node container category', path, node)
    if not isinstance(transformed, (dict, list, str, int, float, bool, type(None))):
        raise _node_error(f'Unsupported node type: {type(transformed)}', path, transformed)
    return transformed


In [ ]:
# | hide
import asyncio
import io
import threading
from contextlib import redirect_stdout
from tempfile import TemporaryDirectory
from unittest.mock import patch

In [ ]:
#| export
from dotenv import load_dotenv

load_dotenv()
OLLAMA_API_KEY= os.getenv('OLLAMA_API_KEY')
DASHSCOPE_API_KEY= os.getenv('DASHSCOPE_API_KEY')

In [ ]:
# | hide
def run_async(coro):
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)

    result = {}
    error = {}

    def _runner():
        try:
            result['value'] = asyncio.run(coro)
        except BaseException as exc:
            error['value'] = exc

    thread = threading.Thread(target=_runner, daemon=True)
    thread.start()
    thread.join()
    if 'value' in error:
        raise error['value']
    return result.get('value')


In [ ]:
# | export
async def get_leaf_summary_async(node:str, min_len:int, client: AsyncClient | AsyncOpenAI, model:str) -> str: # If the string length is less than 200, return the node as is
    """ Generate a summary for a given text node.

    Args:
        node (str): The text node to summarize.
        min_len (int): The minimum length of the text node to summarize.

    Returns:
        str: The summary of the text node.
    """
    
    if len(node) < min_len:
        return node
    # If the node is not an image link, return a summary of the text using unified function
    async with _get_summary_semaphore():
        response_content = await get_summary_response_async(client, node, model=model, role="user", lang='zh')
    if response_content:
        return response_content
    else:
        # If the response is empty, return the original node
        return node


In [ ]:
# | hide
def test_get_leaf_summary_async_threshold_and_fallback():
    async def fake_summary(client, content, model, role, lang):
        return f'SUM<{content}>'

    with patch.dict(globals(), {'get_summary_response_async': fake_summary}):
        test_eq(run_async(get_leaf_summary_async('hi', 5, object(), 'm')), 'hi')
        test_eq(run_async(get_leaf_summary_async('hello', 3, object(), 'm')), 'SUM<hello>')

    async def empty_summary(client, content, model, role, lang):
        return ''

    with patch.dict(globals(), {'get_summary_response_async': empty_summary}):
        test_eq(run_async(get_leaf_summary_async('hello', 3, object(), 'm')), 'hello')


test_get_leaf_summary_async_threshold_and_fallback()

In [ ]:
# | export
async def get_list_summary_async(root: list, leaf_min_len: int, client: AsyncClient | AsyncOpenAI, model:str) -> str:
    """
    Given a list of strings, return a summary of the list.
    If the list is empty, return an empty string.
    If the list has only one element, return the summary of that element.
    If the list has more than one element, return a summary of the concatenated elements.
    """
    list_summary = []
    for n in root:
        if isinstance(n, str):
            # If the element is a string, get its summary
            summary = await get_leaf_summary_async(n, min_len=leaf_min_len, client=client, model=model)
        elif isinstance(n, int) or isinstance(n, float):
            # If the element is a number, get its summary
            summary = await get_leaf_summary_async(str(n), min_len=leaf_min_len, client=client, model=model)
        elif isinstance(n, dict):
            # If the dict has a 's' key, use it as the summary
            summary = n.get('s', '')
        elif isinstance(n, list):
            # If the element is a list, get its summary
            summary = await get_list_summary_async(n, leaf_min_len=leaf_min_len, client=client, model=model)
        elif n is None:
            # If the element is None, skip it
            summary = ""
        else:
            raise ValueError(f"Unsupported element type: {type(n)} in {n}")
        list_summary.append(summary)
        
    # Concatenate all elements and summarize
    # concatenated = " ".join(list_summary)
    # response = get_text_summary_response(concatenated, model="gemma3:27b", role="user", lang='zh')
    # if response.message.content:
    #     return response.message.content
    # else:
    #     # If the response is empty, raise an exception
    #     raise ValueError("Summary response is empty. Please check the input data.")
    return await get_leaf_summary_async(" ".join(list_summary), min_len=leaf_min_len, client=client, model=model)


In [ ]:
# | hide
def test_get_list_summary_async_handles_supported_values():
    async def identity_leaf(node, min_len, client, model):
        return node

    with patch.dict(globals(), {'get_leaf_summary_async': identity_leaf}):
        result = run_async(get_list_summary_async(['a', 2, {'s': 'dict'}, ['nested'], None], 0, object(), 'm'))

    test_eq(result.strip(), 'a 2 dict nested')
    test_fail(lambda: run_async(get_list_summary_async([('bad',)], 0, object(), 'm')), contains='Unsupported element type')


test_get_list_summary_async_handles_supported_values()

In [ ]:
# | export
def _node_error(message: str, path: str, node=None) -> ValueError:
    """Build a ValueError locating a failure by node type and key path within the AST."""
    node_type = node.get("t") if isinstance(node, dict) else type(node).__name__
    details = [message, f"Node type: {node_type}", f"At path: {path}"]
    if node is not None:
        details.insert(1, f"Node: {node}")
    return _LocatedSummaryError("\n".join(details))


def _resolve_image_link(node: dict, root_path: Path, path: str) -> str:
    """Resolve the link of an image node."""
    try:
        image_path = Path(node['c'][2][0])
    except (IndexError, KeyError, TypeError) as exc:
        raise _node_error('Invalid image node structure', path, node) from exc

    if not str(image_path) or not re.search(image_link_pattern, str(image_path)):
        raise _node_error(f'Invalid image link: {image_path}', path, node)
    return str((root_path / image_path).resolve())


In [ ]:
# | hide
def test_resolve_image_link_validates_shape_and_extension():
    root = Path('/tmp/root')
    test_eq(
        _resolve_image_link({'c': [None, None, ['robot.png', '']]}, root, '$'),
        str(root / 'robot.png'),
    )
    test_eq(
        _resolve_image_link({'c': [None, None, ['images/../robot.png', '']]}, Path('docs'), '$'),
        str((Path('docs') / 'robot.png').resolve()),
    )
    test_fail(
        lambda: _resolve_image_link({'c': []}, root, '$.blocks[0].c[1]'),
        contains='Invalid image node structure',
    )
    test_fail(
        lambda: _resolve_image_link({'c': [None, None, ['robot.txt', '']]}, root, '$.blocks[0].c[1]'),
        contains='Invalid image link',
    )
    test_fail(
        lambda: _resolve_image_link({'c': []}, root, '$.blocks[0].c[1]'),
        contains='At path: $.blocks[0].c[1]',
    )


test_resolve_image_link_validates_shape_and_extension()


In [ ]:
# | export
@_with_summary_context('Error summarizing image node')
async def _summarize_image_node_async(node: dict, root_path: Path, client: AsyncClient|AsyncOpenAI, model: str, path: str = "$", lang: str = "zh") -> dict:
    """Summarize an image node asynchronously.

    Args:
        node (dict): The image node.
        root_path (Path): Document root used to resolve the relative image link.
        client (AsyncClient|AsyncOpenAI): Multimodal client used for the image summary.
        model (str): Model name passed to the client.
        path (str): Key path of the node within the AST, used for logging and errors.
        lang (str): Language forwarded to the image-summary provider.

    Returns:
        dict: The summarized image node.
    """
    if not isinstance(node.get('c', []), list):
        raise _node_error("Invalid image caption format", path, node)

    image_link = _resolve_image_link(node, root_path, path)
    async with _get_summary_semaphore():
        node["s"] = await get_image_summary_async(
            client=client,
            image_link=image_link,
            model=model,
            role="user",
            lang=lang
        )
    print(f"Summarize image: {image_link} at {path}")
    return node


In [ ]:
# | hide
def test_summarize_image_node_async_sets_summary_and_validates_caption():
    async def fake_image_summary(client, image_link, model, role, lang):
        return f'IMG<{Path(image_link).name}>'

    with TemporaryDirectory() as tmpdir:
        root = Path(tmpdir)
        node = {'t': 'Image', 'c': [['', [], []], [], ['robot.png', '']]}
        stdout = io.StringIO()
        with patch.dict(globals(), {'get_image_summary_async': fake_image_summary}):
            with redirect_stdout(stdout):
                result = run_async(_summarize_image_node_async(node, root, object(), 'm', path='$.blocks[0].c[1]'))

    test_eq(result['s'], 'IMG<robot.png>')
    test_eq('Summarize image:' in stdout.getvalue(), True)
    test_eq('at $.blocks[0].c[1]' in stdout.getvalue(), True)
    test_fail(lambda: run_async(_summarize_image_node_async({'t': 'Image', 'c': 'bad'}, Path('.'), object(), 'm')), contains='Invalid image caption format')


test_summarize_image_node_async_sets_summary_and_validates_caption()


In [ ]:
# | export
async def _summarize_leaf_async(node: str, leaf_min_len: int, min_len: Optional[int], client: AsyncClient|AsyncOpenAI, model: str, lang: str) -> str:
    """Summarize a leaf node asynchronously."""
    text = str(node).strip()
    threshold = leaf_min_len if min_len is None else min_len
    if len(text) < threshold:
        return text

    async with _get_summary_semaphore():
        response_content = await get_summary_response_async(
            client,
            text,
            model=model,
            role='user',
            lang=lang,
        )
    return response_content or text


In [ ]:
# | hide
def test_summarize_leaf_async_respects_threshold_and_fallback():
    async def fake_summary(client, text, model, role, lang):
        return f'TXT<{text}>'

    with patch.dict(globals(), {'get_summary_response_async': fake_summary}):
        test_eq(run_async(_summarize_leaf_async('hi', 5, None, object(), 'm', 'zh')), 'hi')
        test_eq(run_async(_summarize_leaf_async('hello', 1, None, object(), 'm', 'zh')), 'TXT<hello>')

    async def empty_summary(client, text, model, role, lang):
        return ''

    with patch.dict(globals(), {'get_summary_response_async': empty_summary}):
        test_eq(run_async(_summarize_leaf_async('hello', 1, None, object(), 'm', 'zh')), 'hello')


test_summarize_leaf_async_respects_threshold_and_fallback()

In [ ]:
# | export
async def _coerce_summary_async(value: str|int|float|dict|list|None, leaf_min_len: int, min_len: Optional[int], client: AsyncClient|AsyncOpenAI, model: str, lang: str) -> str:
    """Coerce a value into a summarized string.

    Args:
        self (DOMClass): The DOMClass instance.
        value: The value to be coerced into a summarized string.

    Raises:
        ValueError: If the value type is unsupported.

    Returns:
        str: The summarized string.
    """
    if isinstance(value, str):
        return await _summarize_leaf_async(value, leaf_min_len=leaf_min_len, min_len=min_len, client=client, model=model, lang=lang)
    if isinstance(value, (int, float)):
        return await _summarize_leaf_async(str(value), leaf_min_len=leaf_min_len, min_len=min_len, client=client, model=model, lang=lang)
    if isinstance(value, dict):
        return value.get('s', '')
    if isinstance(value, list):
        return await _summarize_list_async(value, leaf_min_len=leaf_min_len, min_len=min_len, client=client, model=model, lang=lang)
    if value is None:
        return ""
    raise ValueError(f"Unsupported element type: {type(value)} in {value}")


In [ ]:
# | hide
def test_coerce_summary_async_handles_supported_shapes():
    async def fake_leaf(node, leaf_min_len, min_len, client, model, lang):
        return f'LEAF<{node}>'

    async def fake_list(value, leaf_min_len, min_len, client, model, lang):
        return f'LIST<{len(value)}>'

    with patch.dict(globals(), {'_summarize_leaf_async': fake_leaf, '_summarize_list_async': fake_list}):
        test_eq(run_async(_coerce_summary_async('x', 1, 2, object(), 'm', 'zh')), 'LEAF<x>')
        test_eq(run_async(_coerce_summary_async(2, 1, 2, object(), 'm', 'zh')), 'LEAF<2>')
        test_eq(run_async(_coerce_summary_async({'s': 'dict'}, 1, 2, object(), 'm', 'zh')), 'dict')
        test_eq(run_async(_coerce_summary_async(['a', 'b'], 1, 2, object(), 'm', 'zh')), 'LIST<2>')
        test_eq(run_async(_coerce_summary_async(None, 1, 2, object(), 'm', 'zh')), '')

    test_fail(lambda: run_async(_coerce_summary_async({1, 2}, 1, 2, object(), 'm', 'zh')), contains='Unsupported element type')


In [ ]:
# | export
async def _summarize_list_async(
    root: list,
    leaf_min_len: int,
    min_len: Optional[int],
    client: AsyncClient|AsyncOpenAI,
    model: str,
    lang: str
) -> str:
    """Summarize a list of values asynchronously."""
    async def coerce_item(index, item):
        return await _coerce_summary_async(item, leaf_min_len=leaf_min_len, min_len=min_len, client=client, model=model, lang=lang)
    parts = await _ordered_summary_map(coerce_item, root)
    merged = ' '.join(part for part in parts if part)
    return await _summarize_leaf_async(merged, leaf_min_len=leaf_min_len, min_len=min_len, client=client, model=model, lang=lang)


In [ ]:
# | hide
def test_summarize_list_async_merges_nonempty_parts():
    async def fake_coerce(item, leaf_min_len, min_len, client, model, lang):
        return {'a': 'A', 'b': '', 'c': 'C'}[item]

    async def fake_leaf(node, leaf_min_len, min_len, client, model, lang):
        return f'SUM<{node}>'

    with patch.dict(globals(), {'_coerce_summary_async': fake_coerce, '_summarize_leaf_async': fake_leaf}):
        result = run_async(_summarize_list_async(['a', 'b', 'c'], 0, None, object(), 'm', 'zh'))

    test_eq(result, 'SUM<A C>')


test_coerce_summary_async_handles_supported_shapes()
test_summarize_list_async_merges_nonempty_parts()

In [ ]:
# | export
def _record_summary_progress(node_type: str, path: str, file_path: Path, table_count: dict, section_count: dict) -> None:
    """Record the progress of summarizing a node.

    Args:
        node_type (str): The type of the node being summarized.
        path (str): Key path of the node within the AST.
        file_path (Path): Path of the document being summarized.
        table_count (dict): Mutable cumulative table counter ({"count": int}), incremented in place.
        section_count (dict): Mutable cumulative section counter ({"count": int}), incremented in place.
    """
    if node_type == "Table":
        table_count["count"] += 1
        print(f"{file_path} Summarize table: {table_count['count']} at {path}")
    elif node_type == "Section":
        section_count["count"] += 1
        print(f"{file_path} Summarize section: {section_count['count']} at {path}")


In [ ]:
# | hide
def test_record_summary_progress_prints_expected_labels():
    stdout = io.StringIO()
    table_counter = {"count": 1}
    section_counter = {"count": 2}
    with redirect_stdout(stdout):
        _record_summary_progress('Table', '$.blocks[0].c[1]', Path('doc.md'), table_counter, section_counter)
        _record_summary_progress('Section', '$.blocks[0]', Path('doc.md'), table_counter, section_counter)

    output = stdout.getvalue()
    test_eq('Summarize table: 2 at $.blocks[0].c[1]' in output, True)
    test_eq('Summarize section: 3 at $.blocks[0]' in output, True)
    test_eq(table_counter["count"], 2)
    test_eq(section_counter["count"], 3)


test_record_summary_progress_prints_expected_labels()


In [ ]:
# | export
@_with_summary_context('Error summarizing node')
async def summarize_node_async(
    node: dict | list | str | int | float | bool | None,
    root_path: Path,
    client: AsyncClient | AsyncOpenAI,
    model: str,
    file_path: Path,
    table_count: dict,
    section_count: dict,
    path: str = '$',
    action: Optional[Callable] = None,
    leaf_min_len: int = 100,
    min_len: Optional[int] = None,
    lang: str = 'zh',
) -> dict | list | str | int | float | bool | None:
    """Walk one AST with bounded ordered work and awaited sibling cleanup.

    Actions run once on traversed containers/scalars except t/s and image/ignored
    node internals. Replacements preserve container category. Callbacks must avoid
    shared order-dependent state. Scalar coercion retains the legacy text contract.
    """
    node = _apply_summary_action(node, action, path)
    if isinstance(node, dict):
        node_type = node.get('t')
        if node_type is None:
            raise _node_error("Node does not have a 't' key", path, node)
        if node_type == 'Div' and 'object' in node:
            await _summarize_markdown_object_async(
                node['object'], client, model, lang, _object_summary_progress.get(), require_summary=True)
            node['s'] = node['object']['summary']
            if node['object']['type'] == 'table':
                node['subnodes'] = _row_summary_nodes(node['object'])
            return node
        if node_type == 'Image' and not node.get('c'):
            node['s'] = ''
            return node
        if node_type == 'Image':
            return await _summarize_image_node_async(node, root_path, client, model, path, lang)
        if node_type in {'Cite', 'AlignDefault', 'ColWidth'}:
            node['s'] = ''
            return node
        parts = []
        for key, value in list(node.items()):
            if key in {'t', 's'}:
                continue
            child_path = _summary_child_path(path, key)
            if isinstance(value, list):
                items = _apply_summary_action(value, action, child_path)

                async def summarize_item(index, item):
                    return await summarize_node_async(
                        item, root_path, client, model, file_path, table_count, section_count,
                        _summary_child_path(child_path, index), action, leaf_min_len, min_len, lang)

                children = await _ordered_summary_map(summarize_item, items)
                node[key] = children
                if children:
                    parts.append(await _summarize_list_async(children, leaf_min_len, min_len, client, model, lang))
            elif isinstance(value, dict):
                child = await summarize_node_async(
                    value, root_path, client, model, file_path, table_count, section_count,
                    child_path, action, leaf_min_len, min_len, lang)
                node[key] = child
                parts.append(child.get('s', ''))
            else:
                value = _apply_summary_action(value, action, child_path)
                node[key] = value
                if value is not None and value != '':
                    parts.append(await _summarize_leaf_async(str(value), leaf_min_len, min_len, client, model, lang))
        merged = ' '.join(part for part in parts if part)
        node['s'] = await _summarize_leaf_async(merged, leaf_min_len, min_len, client, model, lang) if node.get('c') and merged else ''
        _record_summary_progress(node_type, path, file_path, table_count, section_count)
    elif isinstance(node, list):
        async def summarize_item(index, item):
            item_path = _summary_child_path(path, index)
            if isinstance(item, (dict, list)):
                return await summarize_node_async(
                    item, root_path, client, model, file_path, table_count, section_count,
                    item_path, action, leaf_min_len, min_len, lang)
            item = _apply_summary_action(item, action, item_path)
            return str(item).strip() if item is not None else None
        node = await _ordered_summary_map(summarize_item, node)
    elif node is not None:
        node = await _summarize_leaf_async(str(node), leaf_min_len, min_len, client, model, lang)
    return node


In [ ]:
# | hide
def test_summarize_node_async_summarizes_dict_list_and_scalar_paths():
    payload = {'t': 'Section', 'c': [{'t': 'Header', 'c': [2]}, 'hello', {'t': 'Cite', 'c': []}]}
    progress = []

    async def fake_leaf(node, leaf_min_len, min_len, client, model, lang):
        return f'S<{node}>'

    async def fake_list(root, leaf_min_len, min_len, client, model, lang):
        return f'L<{len(root)}>'

    def fake_progress(node_type, path, file_path, table_count, section_count):
        progress.append((node_type, path, table_count["count"], section_count["count"]))

    with patch.dict(globals(), {'_summarize_leaf_async': fake_leaf, '_summarize_list_async': fake_list, '_record_summary_progress': fake_progress}):
        result = run_async(
            summarize_node_async(json.loads(json.dumps(payload)), Path('.'), client=object(), model='m', file_path=Path('doc.md'), table_count={"count": 1}, section_count={"count": 2}, leaf_min_len=0, min_len=None)
        )

    test_eq(result['c'][0]['s'], 'S<L<1>>')
    test_eq(result['c'][1], 'S<hello>')
    test_eq(result['c'][2]['s'], '')
    test_eq(result['s'], 'S<L<3>>')
    test_eq(progress, [('Header', '$.c[0]', 1, 2), ('Section', '$', 1, 2)])
    test_fail(lambda: run_async(summarize_node_async({}, Path('.'), object(), 'm', Path('doc.md'), {"count": 0}, {"count": 0})), contains="Node does not have a 't' key")


def test_summarize_node_async_reports_key_path_in_errors():
    payload = {'t': 'Section', 'c': [{'c': []}]}  # the child has no 't' key

    with patch.dict(globals(), {'_record_summary_progress': lambda *a, **k: None}):
        test_fail(
            lambda: run_async(summarize_node_async(payload, Path('.'), object(), 'm', Path('doc.md'), {"count": 0}, {"count": 0}, path='$.blocks[0]')),
            contains='At path: $.blocks[0].c[0]',
        )


def test_summarize_node_async_default_leaf_min_len_avoids_llm_calls():
    calls = []

    async def counting_summary(client, text, model, role, lang):
        calls.append(text)
        return 'LLM'

    with patch.dict(globals(), {'get_summary_response_async': counting_summary}):
        result = run_async(
            summarize_node_async('short text', Path('.'), client=object(), model='m', file_path=Path('doc.md'), table_count={"count": 0}, section_count={"count": 0})
        )

    test_eq(result, 'short text')
    test_eq(calls, [])


def test_summarize_node_async_gathers_siblings_concurrently():
    payload = {'t': 'Section', 'c': [
        {'t': 'Para', 'c': ['alpha']},
        {'t': 'Para', 'c': ['beta']},
        {'t': 'Para', 'c': ['gamma']},
    ]}

    async def scenario():
        saturated, release = asyncio.Event(), asyncio.Event()
        inflight = {'now': 0, 'max': 0}
        async def leaf(node, *args, **kwargs):
            inflight['now'] += 1
            inflight['max'] = max(inflight['max'], inflight['now'])
            if inflight['now'] >= 2:
                saturated.set()
            try:
                await release.wait()
                return f'S<{node}>'
            finally:
                inflight['now'] -= 1
        async def fake_list(root, *args, **kwargs):
            return f'L<{len(root)}>'
        with patch.dict(globals(), {'_summarize_leaf_async': leaf, '_summarize_list_async': fake_list,
                                    '_SUMMARY_TASK_LIMIT': 3, '_record_summary_progress': lambda *a, **k: None}):
            task = asyncio.create_task(summarize_node_async(
                payload, Path('.'), object(), 'm', Path('doc.md'), {'count': 0}, {'count': 0}))
            try:
                await asyncio.wait_for(saturated.wait(), 2)
                release.set()
                result = await asyncio.wait_for(task, 2)
            finally:
                release.set()
                task.cancel()
                await asyncio.gather(task, return_exceptions=True)
        test_eq(inflight['max'] > 1, True)
        test_eq([child['c'][0] for child in result['c']], ['S<alpha>', 'S<beta>', 'S<gamma>'])
        test_eq([child['s'] for child in result['c']], ['S<L<1>>'] * 3)
        test_eq(result['s'], 'S<L<3>>')
    run_async(scenario())


def test_summarize_leaf_async_caps_llm_fanout_with_semaphore():
    async def scenario():
        saturated, release = asyncio.Event(), asyncio.Event()
        inflight = {'now': 0, 'max': 0}
        async def tracking_summary(client, text, **kwargs):
            inflight['now'] += 1
            inflight['max'] = max(inflight['max'], inflight['now'])
            if inflight['now'] == _SUMMARY_CONCURRENCY:
                saturated.set()
            try:
                await release.wait()
                return f'TXT<{text}>'
            finally:
                inflight['now'] -= 1
        async def fan_out():
            return await asyncio.gather(*(
                _summarize_leaf_async('long enough text', 1, None, object(), 'm', 'zh')
                for _ in range(_SUMMARY_CONCURRENCY * 3)))
        with patch.dict(globals(), {'get_summary_response_async': tracking_summary}):
            task = asyncio.create_task(fan_out())
            try:
                await asyncio.wait_for(saturated.wait(), 2)
                release.set()
                results = await asyncio.wait_for(task, 2)
            finally:
                release.set()
                task.cancel()
                await asyncio.gather(task, return_exceptions=True)
        test_eq(len(results), _SUMMARY_CONCURRENCY * 3)
        test_eq(results[0], 'TXT<long enough text>')
        test_eq(inflight['max'], _SUMMARY_CONCURRENCY)
    run_async(scenario())


def test_summarize_node_async_threads_action_through_every_depth():
    payload = {'t': 'Section', 'c': [{'t': 'Para', 'c': ['alpha'], 'subnodes': {'t': 'Str', 'c': 'x'}}]}
    visited = []

    def tag(node):
        visited.append(node.get('t') if isinstance(node, dict) else type(node).__name__)
        return node

    async def fake_summary(client, text, model, role, lang):
        return f'[{text}]'

    with patch.dict(globals(), {'get_summary_response_async': fake_summary, '_record_summary_progress': lambda *a, **k: None}):
        run_async(
            summarize_node_async(payload, Path('.'), client=object(), model='m', file_path=Path('doc.md'), table_count={"count": 0}, section_count={"count": 0}, action=tag, leaf_min_len=0, min_len=0)
        )

    test_eq(visited, ['Section', 'list', 'Para', 'list', 'str', 'Str', 'str'])


test_summarize_node_async_summarizes_dict_list_and_scalar_paths()
test_summarize_node_async_reports_key_path_in_errors()
test_summarize_node_async_default_leaf_min_len_avoids_llm_calls()
test_summarize_node_async_gathers_siblings_concurrently()
test_summarize_leaf_async_caps_llm_fanout_with_semaphore()
test_summarize_node_async_threads_action_through_every_depth()


In [ ]:
# | export
async def summary_node_async(
        node: dict | list,
        root_path: Path,
        file_path: Path,
        client: AsyncClient|AsyncOpenAI,
        model: str,
        table_count: int,
        section_count: int,
        action: Optional[Callable] = None,
        leaf_min_len: int = 0,
        min_len: Optional[int] = None,
        lang: str='zh',
    ) -> dict | list:
    '''
    Given a string node, add key,value pair: node['s'] = node_summary, and return the node 
    '''

    if isinstance(node, dict):
        try:
            t = node["t"]
        except KeyError:
            raise ValueError(f"Node does not have a 't' key: {node}")
        if t == "Image":  # Image summary, the Image node is as defined in the pandoc AST
            summary = []
            if (not node['c'][1] == []) and (node['c'][1][0] is not None) and (node['c'][1][0].get('c') is not None):
                summary.append(node['c'][1][0]['c'])  # The content of the second element is the image caption
            try:
                # summary.append(node['c'][0])  # The first element are defined to be attributes of the image rendering, i.e. content-irrelevant.
                # If the node is an image, get its link
                image_link = root_path / node['c'][2][0]  # Assuming the image link is in the third element of the list
                image_link = str(image_link)
                if image_link and re.search(image_link_pattern, image_link):  # re.match leads to empty match if there's spaces in the path!
                    # If the node is an image link, get its summary
                    assert isinstance(client,AsyncClient), "Only ollama client is allowed for image summary"
                    async with _get_summary_semaphore():
                        response = await get_image_summary_response_async(
                            client=client,
                            image_link=image_link,
                            model=model,
                            role="user",
                            lang=lang
                            )
                    summary.append(response.message.content)
                else:
                    # If the node is not an image link, summarize its content
                    raise ValueError(f"Invalid image link: {image_link}")
                # Get the summary of the image caption
            except (IndexError, KeyError):
                # Handle cases where the image link is not in the expected format
                raise ValueError(f"Invalid image node structure: {node}")
            # The second element is the image title
            summary.append(node['c'][2][1])  # The second element is the image title
            assert isinstance(client,AsyncClient), "Only ollama client is allowed for image summary"
            async with _get_summary_semaphore():
                response_txt = await get_text_summary_response_async(
                    client=client,
                    content=" ".join(summary),
                    model=model,
                    role="user",
                    lang=lang
                )
            node["s"] = response_txt.message.content
            print(f"Summarize image: {image_link}")
        elif t == "Cite" or t == "AlignDefault" or t == "ColWidth" :  # Quoted node will be ignored
            node["s"] = ""  # Set the summary to an empty string
        else: # TextBlock summary
            c = node.get("c", None)
            if not c:
                node["s"] = ""
            else:
                dict_summary = []
                for key, value in node.items(): # get summary of the values (content)
                    if key == "t":
                        continue
                    if isinstance(value, list):  # get the summary of the string list
                        if value == []:
                            # If the list is empty, skip it
                            continue
                        # If the value is a list, summarize each element
                        node[key] = [
                            await summary_node_async(
                                node=child,
                                root_path=root_path,
                                file_path=file_path,
                                client=client,
                                model=model,
                                table_count=table_count,
                                section_count=section_count,
                                action=action,
                                leaf_min_len=leaf_min_len,
                                min_len=min_len,
                                lang=lang
                                )
                            if isinstance(child, (dict,list))
                            else child
                            for child in value
                        ]
                        dict_summary.append(
                            await _summarize_list_async(
                                root=value,
                                leaf_min_len=leaf_min_len,
                                min_len=min_len,
                                client=client,
                                model=model,
                                lang=lang)
                            )
                    elif isinstance(value, dict):
                        child = await summary_node_async(
                            node=value,
                            root_path=root_path,
                            file_path=file_path,
                            client=client,
                            model=model,
                            table_count=table_count,
                            section_count=section_count,
                            action=action,
                            leaf_min_len=leaf_min_len,
                            min_len=min_len,
                            lang=lang
                        )  # insert the value['s']
                        assert isinstance(child, dict) and 's' in child, f"Expected dict with 's' key, got {child}"
                        dict_summary.append(child['s'])
                    elif value is None or value == "":
                        # If the value is None, skip it
                        continue
                    else:
                        if not isinstance(value, str):
                            # If the value is not a string, convert it to a string
                            value = str(value)
                        dict_summary.append(
                            await _summarize_leaf_async(
                                node=value,
                                leaf_min_len=leaf_min_len,
                                min_len=min_len,
                                client=client,
                                model=model,
                                lang=lang
                                )
                            )
                # get the summary of the node
                if dict_summary:  # type: ignore
                    # If there are summaries, concatenate them
                    node["s"] = await _summarize_leaf_async(
                        node=" ".join(dict_summary),
                        leaf_min_len=leaf_min_len,
                        min_len=min_len,
                        client=client,
                        model=model,
                        lang=lang
                    )
                    # node["s"] = get_text_summary_response(
                    #     " ".join(dict_summary), model="gemma3:27b", role="user", lang='zh'
                    # ).message.content
                else:
                    # If no summaries, set to empty string
                    node["s"] = ""
            # if t is table
            if t == "Table":
                print(f"{file_path} Summarize table: {table_count}")
                table_count += 1
            elif t == "Section":
                print(f"{file_path} Summarize section: {section_count} section depth {node['c'][0]['c'][0]}")
                section_count += 1
    elif isinstance(node, list):
        node = [
            await summary_node_async(
                node=child,
                root_path=root_path,
                file_path=file_path,
                client=client,
                model=model,
                table_count=table_count,
                section_count=section_count,
                action=action,
                leaf_min_len=leaf_min_len,
                min_len=min_len,
                lang=lang
                ) if isinstance(child, (dict,list)) else child
            for child in node
        ]
    else:
        assert isinstance(node, str), f"Expected a string node, got {type(node)}, on json line"
        # If the node is a string, get its summary
        node_summary = await _summarize_leaf_async(
            node=node,
            leaf_min_len=leaf_min_len,
            min_len=min_len,
            client=client,
            model=model,
            lang=lang
        )
        node = node_summary
    return node

In [ ]:
# | hide
def test_summary_node_async_summarizes_legacy_node_shapes():
    payload = {'t': 'Section', 'c': [{'t': 'Header', 'c': [2]}, 'hello', {'t': 'Cite', 'c': []}]}

    async def fake_leaf(node, leaf_min_len, min_len, client, model, lang):
        return f'S<{node}>'

    async def fake_list(root, leaf_min_len, min_len, client, model, lang):
        return 'LIST'

    with patch.dict(globals(), {'_summarize_leaf_async': fake_leaf, '_summarize_list_async': fake_list}):
        result = run_async(summary_node_async(json.loads(json.dumps(payload)), Path('.'), Path('doc.md'), object(), 'm', 1, 2))

    test_eq(result['s'], 'S<LIST>')
    test_eq(result['c'][0]['s'], 'S<LIST>')
    test_eq(result['c'][1], 'hello')
    test_eq(result['c'][2]['s'], '')
    test_fail(lambda: run_async(summary_node_async({}, Path('.'), Path('doc.md'), object(), 'm', 0, 0)), contains="Node does not have a 't' key")


test_summary_node_async_summarizes_legacy_node_shapes()

## Markdown object trees and summarization

Full document trees retain prose and recursive sections. Figures, HTML and Markdown pipe tables, and lists keep their original subtrees, source records, and adjacent captions inside atomic `Div` nodes. Flat extraction and summarization remain available.


In [ ]:
# | export
from html.parser import HTMLParser
from urllib.parse import unquote, urlsplit
from markdown_it import MarkdownIt
from tempfile import NamedTemporaryFile
from uuid import uuid4
import pypandoc

from ribosome.core.dom.reorg import reorg_node

_object_summary_progress = ContextVar("object_summary_progress", default=None)


def _inline_text(tokens):
    return ''.join(
        _inline_text(t.children) if t.children else
        t.content if t.type in {'text', 'code_inline', 'image'} else
        ' ' if t.type in {'softbreak', 'hardbreak'} else ''
        for t in tokens
    ).strip()


class _TableSpans(HTMLParser):
    """Locate outer HTML tables without rewriting their source."""
    def __init__(self, source):
        super().__init__(convert_charrefs=False)
        self.source = source
        self.offsets = [0]
        for line in source.splitlines(keepends=True):
            self.offsets.append(self.offsets[-1] + len(line))
        self.depth = 0
        self.start = None
        self.spans = []
        self.feed(source)
        self.close()
        if self.depth:
            raise ValueError('Unclosed HTML table')

    def _offset(self):
        line, col = self.getpos()
        return self.offsets[line - 1] + col

    def handle_starttag(self, tag, attrs):
        if tag == 'table':
            if self.depth == 0:
                self.start = self._offset()
            self.depth += 1

    def handle_endtag(self, tag):
        if tag == 'table' and self.depth:
            self.depth -= 1
            if self.depth == 0:
                end = self.source.index('>', self._offset()) + 1
                self.spans.append((self.start, end))


class _HTMLTableRows(HTMLParser):
    """Read only outer-table rows, retaining source slices and cell spans."""
    def __init__(self, source):
        super().__init__(convert_charrefs=True)
        self.source, self.rows = source, []
        self.offsets = [0]
        for line in source.splitlines(keepends=True):
            self.offsets.append(self.offsets[-1] + len(line))
        self.depth = 0
        self.section, self.section_id = '', 0
        self.row = self.cell = None
        self.feed(source)
        self.close()
        self._finish_row(len(source))

    def _offset(self):
        line, column = self.getpos()
        return self.offsets[line - 1] + column

    def _finish_cell(self, end):
        if self.cell is not None:
            self.cell['content'] = self.source[self.cell.pop('_a'):end]
            self.cell['text'] = ' '.join(''.join(self.cell.pop('_text')).split())
            self.row['cells'].append(self.cell)
            self.cell = None

    def _finish_row(self, end):
        if self.row is not None:
            self._finish_cell(end)
            self.row['_b'] = end
            self.row['is_header'] = self.section == 'thead' or (
                bool(self.row['cells']) and all(c['is_header'] for c in self.row['cells']))
            self.rows.append(self.row)
            self.row = None

    def handle_starttag(self, tag, attrs):
        offset = self._offset()
        if tag == 'table':
            self.depth += 1
        if self.depth == 1:
            if tag in {'thead', 'tbody', 'tfoot'}:
                self._finish_row(offset)
                self.section, self.section_id = tag, self.section_id + 1
            elif tag == 'tr':
                self._finish_row(offset)
                self.row = {'_a': offset, 'cells': [], '_section': self.section_id}
            elif tag in {'td', 'th'} and self.row is not None:
                self._finish_cell(offset)
                attrs = dict(attrs)
                def span(name):
                    try:
                        value = int(attrs.get(name, '1'))
                        return value if value > 0 or (name == 'rowspan' and value == 0) else 1
                    except (ValueError, TypeError):
                        return 1
                self.cell = {'_a': offset, '_text': [], 'is_header': tag == 'th',
                             'rowspan': span('rowspan'), 'colspan': span('colspan')}
        if self.cell is not None and tag in {'br', 'p', 'div', 'li', 'table', 'tr', 'td', 'th'}:
            self.cell['_text'].append(' ')
        if self.cell is not None and tag == 'img':
            self.cell['_text'].append(dict(attrs).get('alt') or '')

    def handle_endtag(self, tag):
        offset = self._offset()
        end = self.source.index('>', offset) + 1
        if self.depth == 1:
            if tag in {'td', 'th'}:
                self._finish_cell(end)
            elif tag == 'tr':
                self._finish_row(end)
            elif tag in {'thead', 'tbody', 'tfoot', 'table'}:
                self._finish_row(offset)
                self.section, self.section_id = '', self.section_id + 1
        if self.cell is not None and tag in {'p', 'div', 'li', 'table', 'tr', 'td', 'th'}:
            self.cell['_text'].append(' ')
        if tag == 'table':
            self.depth -= 1

    def handle_data(self, data):
        if self.cell is not None:
            self.cell['_text'].append(data)


def _table_group_ranges(ranges):
    """Coalesce identical/crossing spans while retaining contained subgroups."""
    ranges = sorted(set(ranges))
    parents = list(range(len(ranges)))

    def root(index):
        while parents[index] != index:
            parents[index] = parents[parents[index]]
            index = parents[index]
        return index

    # Crossing intervals cannot be siblings without duplicating physical rows.
    # Collapse their connected components; compatible contained spans survive.
    for i, (start, end) in enumerate(ranges):
        for j in range(i + 1, len(ranges)):
            next_start, next_end = ranges[j]
            if next_start >= end:
                break
            if start < next_start < end < next_end:
                parents[root(j)] = root(i)
    bounds = {}
    for i, (start, end) in enumerate(ranges):
        component = root(i)
        low, high = bounds.get(component, (start, end))
        bounds[component] = (min(low, start), max(high, end))
    return sorted(set(bounds.values()), key=lambda span: (span[0], -span[1]))


def _table_row_records(obj, content_line_start, env=None):
    """Build rows and nested merged groups without modifying table source."""
    source = obj['content']
    html = source.lstrip().lower().startswith('<table')
    rows, ranges = [], []
    if html:
        rows = _HTMLTableRows(source).rows
        # Vertical spans stop at row-group boundaries, including rowspan=0.
        ends = {}
        for i, row in enumerate(rows):
            ends[row['_section']] = i + 1
        active = []
        for i, row in enumerate(rows):
            active = [cell for cell in active if cell['source_row'] - 1 + cell['rowspan'] > i]
            occupied = {col for cell in active
                        for col in range(cell['column'], cell['column'] + cell['colspan'])}
            cells = [dict(cell, inherited=True) for cell in active]
            column = 1
            for cell in row['cells']:
                while any(col in occupied for col in range(column, column + cell['colspan'])):
                    column += 1
                remaining = ends[row['_section']] - i
                cell['rowspan'] = min(cell['rowspan'] or remaining, remaining)
                cell.update(column=column, source_row=i + 1, inherited=False)
                cells.append(cell)
                if cell['rowspan'] > 1:
                    active.append(cell)
                    ranges.append((i, i + cell['rowspan']))
                occupied.update(range(column, column + cell['colspan']))
                column += cell['colspan']
            row['cells'] = sorted(cells, key=lambda cell: cell['column'])
    else:
        lines = source.splitlines(keepends=True)
        offsets = [0]
        for line in lines:
            offsets.append(offsets[-1] + len(line))
        tokens = MarkdownIt('commonmark', {'html': True}).enable('table').parse(source, env)
        row, header = None, False
        for token in tokens:
            if token.type == 'thead_open':
                header = True
            elif token.type == 'thead_close':
                header = False
            elif token.type == 'tr_open':
                start, end = token.map
                row = {'_a': offsets[start], '_b': offsets[end],
                       'cells': [], 'is_header': header}
                rows.append(row)
            elif token.type == 'inline' and row is not None:
                row['cells'].append({'content': token.content, 'text': _inline_text(token.children or []),
                                     'column': len(row['cells']) + 1, 'source_row': len(rows),
                                     'rowspan': 1, 'colspan': 1, 'is_header': header, 'inherited': False})
            elif token.type == 'tr_close':
                row = None

    def record(kind, identity, a, b, **fields):
        return dict(id=f"{obj['id']}.{identity}", type=kind, table_id=obj['id'],
                    source_file=obj['source_file'],
                    line_start=content_line_start + source[:a].count('\n'),
                    line_end=content_line_start + source[:b].rstrip('\n').count('\n'),
                    content=source[a:b], caption=obj['caption'], images=[], summary='',
                    children=[], **fields)

    records = [record('table_row', f'row.{i + 1}', row['_a'], row['_b'],
                      row_index=i + 1, is_header=row['is_header'], cells=row['cells'])
               for i, row in enumerate(rows)]
    headers = [row['cells'] for row in records if row['is_header']]
    children = []
    # Each frame holds the range end, next unassigned row, and output children.
    stack = [[len(rows), 0, children]]
    for start, end in _table_group_ranges(ranges):
        while stack[-1][0] <= start:
            group_end, position, output = stack.pop()
            output.extend(records[position:group_end])
        parent = stack[-1]
        parent[2].extend(records[parent[1]:start])
        group = record('table_row_group', f'group.{start + 1}-{end}',
                       rows[start]['_a'], rows[end - 1]['_b'],
                       row_start=start + 1, row_end=end)
        parent[2].append(group)
        parent[1] = end
        stack.append([end, start, group['children']])
    while stack:
        group_end, position, output = stack.pop()
        output.extend(records[position:group_end])
    return children, headers


def _row_summary_nodes(obj):
    """Mirror semantic children as Divs, leaving the original Pandoc c intact."""
    return [{'t': 'Div', 'c': [['', [], []], []], 'object': child,
             's': child['summary'], 'subnodes': _row_summary_nodes(child)}
            for child in obj.get('children', [])]


async def _summarize_table_child_async(row, table, client, model, lang):
    try:
        async def summarize_child(index, child):
            return await _summarize_table_child_async(child, table, client, model, lang)
        await _ordered_summary_map(summarize_child, row['children'])
        payload = {'caption': table['caption'], 'headers': table['headers'],
                   'content': row['content']}
        if row['type'] == 'table_row':
            instruction = 'Summarize this table row, preserving column relationships and inherited merged-cell values.'
            payload.update(row_index=row['row_index'], is_header=row['is_header'], cells=row['cells'])
        else:
            instruction = ('Summarize this merged table row group using every immediate child row '
                           'or subgroup summary and inherited merged-cell values.')
            first_row = row
            while first_row['type'] == 'table_row_group':
                first_row = first_row['children'][0]
            payload.update(row_start=row['row_start'], row_end=row['row_end'],
                           inherited_cells=[cell for cell in first_row['cells'] if cell['inherited']])
            payload['child_summaries'] = [child['summary'] for child in row['children']]
        prompt = instruction + '\n' + json.dumps(payload, ensure_ascii=False)
        async with _get_summary_semaphore():
            row['summary'] = await get_summary_response_async(
                client=client, content=prompt, model=model, role='user', lang=lang)
        if not isinstance(row['summary'], str) or not row['summary'].strip():
            raise ValueError('Provider returned an empty row summary')
        return row
    except Exception as exc:
        raise ValueError(f"Error summarizing {row['type']} {row['id']} in table {table['id']} at "
                         f"{row['source_file']}:{row['line_start']}-{row['line_end']}: {exc}") from exc


def _extract_markdown_objects_with_spans(file_path: str | Path):
    """Extract ordered figures, HTML/pipe tables and Markdown lists, offline.

    Records contain id, type, source_file, line_start/line_end (one-based,
    inclusive, including captions), content (unaltered object source), caption,
    images ({src, resolved_src, alt}), and summary (empty before summarization).
    Tables also retain headers and ordered children (rows or merged row groups).
    Blank lines do not break caption adjacency. Nested list/table contents stay
    within their enclosing object. The input file is never modified.
    """
    file_path = Path(file_path).resolve()
    source = file_path.read_text(encoding='utf-8')
    lines = source.splitlines(keepends=True)
    offsets = [0]
    for line in lines:
        offsets.append(offsets[-1] + len(line))
    parser = MarkdownIt('commonmark', {'html': True}).enable('table')
    env = {}
    tokens = parser.parse(source, env)
    objects, text_lines = [], {}
    i = 0
    while i < len(tokens):
        token = tokens[i]
        if token.level != 0 or token.map is None:
            i += 1
            continue
        start, end = token.map
        while end > start and not lines[end - 1].strip():
            end -= 1
        content = ''.join(lines[start:end])
        kind, images = None, []
        if token.type in {'ordered_list_open', 'bullet_list_open'}:
            kind = 'list'
        elif token.type == 'table_open':
            kind = 'table'
        elif token.type == 'html_block':
            try:
                spans = _TableSpans(content).spans
            except ValueError as exc:
                raise ValueError(f'{file_path}:{start + 1}: {exc}') from exc
            for a, b in spans:
                objects.append({'type': 'table', '_start': start + content[:a].count('\n'),
                                '_end': start + content[:b].count('\n') + 1,
                                '_a': offsets[start] + a, '_b': offsets[start] + b,
                                'content': content[a:b], 'images': []})
        elif token.type == 'paragraph_open':
            for line_index in range(start, end):
                children = parser.parseInline(lines[line_index], env)[0].children or []
                image_tokens = [t for t in children if t.type == 'image']
                image_only = image_tokens and all(
                    t.type in {'image', 'em_open', 'em_close', 'strong_open', 'strong_close',
                               'softbreak', 'hardbreak'} or
                    (t.type == 'text' and not t.content.strip()) for t in children)
                if image_only:
                    line_images = []
                    for image in image_tokens:
                        src = image.attrGet('src') or ''
                        resolved = src if urlsplit(src).scheme or src.startswith('//') else str(
                            (file_path.parent / unquote(urlsplit(src).path)).resolve())
                        line_images.append({'src': src, 'resolved_src': resolved,
                                            'alt': _inline_text(image.children or []) or image.content})
                    objects.append({'type': 'figure', '_start': line_index, '_end': line_index + 1,
                                    '_a': offsets[line_index], '_b': offsets[line_index + 1],
                                    'content': lines[line_index], 'images': line_images})
                else:
                    text_lines[line_index] = _inline_text(children)
        if kind:
            objects.append({'type': kind, '_start': start, '_end': end,
                            '_a': offsets[start], '_b': offsets[end],
                            'content': content, 'images': images})
        # Skip nested blocks: an outer list is a single object.
        if token.nesting == 1:
            depth, j = 1, i + 1
            while j < len(tokens) and depth:
                depth += tokens[j].nesting
                j += 1
            i = j
        else:
            i += 1
    objects.sort(key=lambda obj: obj['_start'])
    grouped = []
    for obj in objects:
        if (grouped and obj['type'] == grouped[-1]['type'] == 'figure'
                and not ''.join(lines[grouped[-1]['_end']:obj['_start']]).strip()):
            previous = grouped[-1]
            previous['_end'] = obj['_end']
            previous['_b'] = obj['_b']
            previous['content'] = ''.join(lines[previous['_start']:obj['_end']])
            previous['images'].extend(obj['images'])
        else:
            grouped.append(obj)
    consumed = set()
    source_spans = []
    for index, obj in enumerate(grouped, 1):
        start, end = obj.pop('_start'), obj.pop('_end')
        a, b = obj.pop('_a'), obj.pop('_b')
        content_line_start = start + 1
        caption = ''
        candidate = end if obj['type'] == 'figure' else start - 1
        step = 1 if obj['type'] == 'figure' else -1
        while 0 <= candidate < len(lines) and not lines[candidate].strip():
            candidate += step
        text = text_lines.get(candidate, '')
        matches = (text.startswith('图 ') if obj['type'] == 'figure' else
                   text.startswith('表 ') if obj['type'] == 'table' else
                   text.endswith((':', '：')))
        if matches and candidate not in consumed:
            caption = text
            consumed.add(candidate)
            start, end = min(start, candidate), max(end, candidate + 1)
            a, b = min(a, offsets[candidate]), max(b, offsets[candidate + 1])
        obj.update(id=index, source_file=str(file_path), line_start=start + 1,
                   line_end=end, caption=caption, summary='')
        if obj['type'] == 'table':
            obj['children'], obj['headers'] = _table_row_records(obj, content_line_start, env)
        source_spans.append((a, b))
    return source, grouped, source_spans


def extract_markdown_objects(file_path: str | Path) -> list[dict]:
    """Extract ordered object records with table children; source and caches stay untouched.

    Captions, source spans, images and original content are retained. Nested
    tables/lists belong to their enclosing object. Tables expose rows and merged
    groups through children. Summaries are initially empty.
    """
    return _extract_markdown_objects_with_spans(file_path)[1]


async def _summarize_markdown_object_async(obj, client, model, lang, progress=None, *, require_summary=False):
    try:
        if obj['type'] == 'table':
            async def summarize_row(index, row):
                return await _summarize_table_child_async(row, obj, client, model, lang)
            await _ordered_summary_map(summarize_row, obj['children'])
        descriptions = []
        for image in obj['images']:
            async with _get_summary_semaphore():
                description = await get_image_summary_async(
                    client=client, image_link=image['resolved_src'], model=model,
                    role='user', lang=lang)
            descriptions.append({'alt': image['alt'], 'description': description})
        table_format = 'HTML' if obj['content'].lstrip().lower().startswith('<table') else 'Markdown pipe'
        instructions = {
            'figure': 'Summarize this figure using every image description, alt text and the shared caption.',
            'table': f'Summarize this complete {table_format} table, preserving row/column relationships, alignment and merged-cell meaning.',
            'list': 'Summarize this complete list, preserving order and nesting.',
        }
        payload = {'caption': obj['caption'], 'content': obj['content']}
        if obj['type'] == 'table':
            payload['child_summaries'] = [child['summary'] for child in obj['children']]
        if descriptions:
            payload['image_descriptions'] = descriptions
        prompt = instructions[obj['type']] + '\n' + json.dumps(payload, ensure_ascii=False)
        async with _get_summary_semaphore():
            obj['summary'] = await get_summary_response_async(
                client=client, content=prompt, model=model, role='user', lang=lang)
        if (require_summary or obj['type'] == 'table') and (not isinstance(obj['summary'], str) or not obj['summary'].strip()):
            raise ValueError('Provider returned an empty object summary')
        if progress is not None:
            progress.update(1)
        return obj
    except Exception as exc:
        raise ValueError(
            f"Error summarizing {obj['type']} object {obj['id']} at "
            f"{obj['source_file']}:{obj['line_start']}-{obj['line_end']}: {exc}"
        ) from exc


async def summarize_markdown_objects_async(
    file_path: str | Path, client: AsyncClient | AsyncOpenAI, model: str, *,
    lang: str = 'zh', show_progress: bool = False
) -> list[dict]:
    """Summarize extracted objects with bounded requests and ordered results.

    Figures combine individual visual descriptions with their caption and alt
    text. Tables retain complete source and summarize each row and merged group
    before the table itself; lists retain Markdown structure. Failures
    identify the source object; cancellation awaits outstanding work. No output
    artifacts or DOM caches are written. Set show_progress=True for a notebook
    progress bar that advances after each completed object.
    """
    objects = extract_markdown_objects(file_path)
    from tqdm.auto import tqdm
    progress = tqdm(total=len(objects), desc=Path(file_path).stem, unit='object') if show_progress else None

    async def summarize_object(index, obj):
        return await _summarize_markdown_object_async(obj, client, model, lang, progress)

    try:
        return await _ordered_summary_map(summarize_object, objects)
    finally:
        if progress is not None:
            progress.close()


def _object_nodes(tree):
    """Yield top-level object wrappers; row children are counted with their table."""
    if isinstance(tree, dict):
        if tree.get('t') == 'Div' and 'object' in tree:
            yield tree
            return
        for key, value in tree.items():
            if key not in {'object', 's', 'summary'}:
                yield from _object_nodes(value)
    elif isinstance(tree, list):
        for value in tree:
            yield from _object_nodes(value)


def extract_markdown_object_tree(file_path: str | Path) -> dict:
    """Parse a full recursive DOM with object Divs and table rows, without writing files.

    Each Div retains its Pandoc attributes and original block subtree in c;
    object holds the same record returned by extract_markdown_objects, and s
    holds its semantic summary. Table subnodes mirror row/group children.
    Section nesting matches reorg_node.
    """
    file_path = Path(file_path).resolve()
    source, objects, spans = _extract_markdown_objects_with_spans(file_path)
    marker = 'data-ribosome-object-' + uuid4().hex
    # Longer fences cannot be closed by any existing colon fence in the source.
    fence = ':' * max(3, 1 + max((len(m.group()) for m in re.finditer(r':{3,}', source)), default=2))
    marked = source
    for obj, (start, end) in reversed(list(zip(objects, spans))):
        marked = (marked[:start] + f'\n{fence} {{{marker}="{obj["id"]}"}}\n\n'
                  + marked[start:end] + f'\n\n{fence}\n\n' + marked[end:])
    tree = json.loads(pypandoc.convert_text(marked, 'json', 'markdown+fenced_divs'))
    tree = reorg_node(tree, section_level=[])
    found = []

    def attach(node):
        if isinstance(node, dict):
            if node.get('t') == 'Div':
                attrs = node['c'][0][2]
                ids = [int(value) for key, value in attrs if key == marker]
                if ids:
                    object_id = ids[0]
                    node['c'][0][2] = [pair for pair in attrs if pair[0] != marker]
                    node['object'] = objects[object_id - 1]
                    node['s'] = ''
                    if node['object']['type'] == 'table':
                        node['subnodes'] = _row_summary_nodes(node['object'])
                    found.append(object_id)
                    return
            for value in node.values():
                attach(value)
        elif isinstance(node, list):
            for value in node:
                attach(value)

    attach(tree)
    if sorted(found) != [obj['id'] for obj in objects]:
        raise ValueError(f'Object spans were not preserved while parsing {file_path}')
    tree.update(title=file_path.stem, file_path=str(file_path), summary='')
    return tree


async def summarize_markdown_object_tree_async(
    file_path: str | Path, client: AsyncClient | AsyncOpenAI, model: str, *,
    lang: str = 'zh', show_progress: bool = False, leaf_min_len: int = 100,
    min_len: Optional[int] = None,
) -> dict:
    """Summarize a full DOM bottom-up, including table rows and merged groups.

    Original object subtrees are retained; table row summaries appear in
    subnodes. Other nodes recurse through c/subnodes. Returns a document with
    blocks, title, file_path and summary; no files or DOM caches are written.
    """
    file_path = Path(file_path).resolve()
    tree = extract_markdown_object_tree(file_path)
    from tqdm.auto import tqdm
    count = sum(1 for _ in _object_nodes(tree))
    progress = tqdm(total=count, desc=file_path.stem, unit='object') if show_progress else None
    token = _object_summary_progress.set(progress)
    try:
        tree['blocks'] = await summarize_node_async(
            tree['blocks'], file_path.parent, client, model, file_path,
            {'count': 0}, {'count': 0}, path='$.blocks',
            leaf_min_len=leaf_min_len, min_len=min_len, lang=lang)
        parts = [tree['title']] + [block['s'] for block in tree['blocks']
                                  if isinstance(block, dict) and block.get('s')]
        tree['summary'] = await _summarize_leaf_async(
            ' '.join(parts), leaf_min_len, min_len, client, model, lang) if tree['blocks'] else ''
        return tree
    finally:
        _object_summary_progress.reset(token)
        if progress is not None:
            progress.close()


def _write_semantics_tree(file_path, tree):
    """Publish a completed tree atomically; failures retain the previous file."""
    if not isinstance(tree, dict) or not isinstance(tree.get('blocks'), list):
        raise ValueError('Expected a document tree with blocks')
    file_path = Path(file_path)
    serialized = json.dumps(tree, ensure_ascii=False, indent=2) + '\n'
    temporary = None
    try:
        with NamedTemporaryFile(mode='w', encoding='utf-8', dir=file_path.parent,
                                prefix=file_path.name + '.', suffix='.tmp', delete=False) as handle:
            temporary = Path(handle.name)
            handle.write(serialized)
        temporary.replace(file_path)
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)


def _configured_ollama_summary_client():
    """Read the project-root .env for manual object-summary examples."""
    from dotenv import dotenv_values

    configured_root = os.getenv('PROJ_ROOT')
    root = Path(configured_root).expanduser().resolve() if configured_root else next(
        (path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
         if (path / 'pyproject.toml').is_file()), None)
    if root is None or not (root / 'pyproject.toml').is_file():
        raise FileNotFoundError('Cannot locate PROJ_ROOT containing pyproject.toml')
    env_file = root / '.env'
    if not env_file.is_file():
        raise FileNotFoundError(f'Missing configuration: {env_file}')
    settings = dotenv_values(env_file)
    host = (settings.get('OLLAMA_HOST') or '').strip()
    model = (settings.get('OLLAMA_SUMMARY_MODEL') or '').strip()
    if not host or not model:
        raise ValueError('Set OLLAMA_HOST and OLLAMA_SUMMARY_MODEL in PROJ_ROOT/.env')
    api_key = settings.get('OLLAMA_API_KEY')
    headers = {'Authorization': f'Bearer {api_key}'} if api_key else {}
    return AsyncClient(host=host, headers=headers), model


In [ ]:
# | hide
with TemporaryDirectory() as tmp:
    file = Path(tmp) / 'sample.md'
    file.write_text('![a](a.png)\n\n图 1 示例\n\n说明：\n\n- 一\n- 二\n', encoding='utf-8')
    objects = extract_markdown_objects(file)
    test_eq([o['type'] for o in objects], ['figure', 'list'])
    test_eq([o['caption'] for o in objects], ['图 1 示例', '说明：'])


## SX322017 manual preview

The live example requires an image-capable provider and is excluded from automated tests.


In [ ]:
# | notest
sample_file = project_root / 'assets/SX322017-C-1/SX322017《新松机器人跟随应用操作手册》(C-1).md'
object_tree = extract_markdown_object_tree(sample_file)
semantics_file = sample_file.with_name(f'{sample_file.stem}_semantics.json')
# Read-only preview: publish only after all summaries complete successfully.
[(node['object']['id'], node['object']['type'], node['object']['caption'],
  node['object']['line_start'], node['object']['line_end'])
 for node in _object_nodes(object_tree)]


In [ ]:
# | notest
client, summary_model = _configured_ollama_summary_client()
object_summaries = await summarize_markdown_object_tree_async(
    sample_file, client, summary_model, lang='zh', show_progress=True)
assert all(node['s'] and node['s'] == node['object']['summary']
           for node in _object_nodes(object_summaries))
_write_semantics_tree(semantics_file, object_summaries)
print(f'Exported summary tree: {semantics_file}')
object_summaries


## SX322018 functional test

Uses `OLLAMA_HOST`, `OLLAMA_SUMMARY_MODEL`, and optional `OLLAMA_API_KEY` from `PROJ_ROOT/.env`. Run the next two cells to summarize the recursive document with object progress and atomically export `<markdown_stem>_semantics.json` after success. Checks retained object metadata, populated summaries, JSON round-trip, section structure, and unchanged Markdown. Requires an image-capable Ollama model; excluded from automated tests.


In [ ]:
# | notest
async def test_summarize_objects_with_sx322018_md():
    configured_root = os.getenv('PROJ_ROOT')
    root = Path(configured_root).expanduser().resolve() if configured_root else next(
        (path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
         if (path / 'pyproject.toml').is_file()), None)
    assert root is not None and (root / 'pyproject.toml').is_file(), 'Invalid or missing PROJ_ROOT'
    file_path = root / 'assets/SX322018-A-0/SX322018《新松机器人气动点焊应用操作手册》-A-0.md'
    assert file_path.is_file(), f'Markdown source missing: {file_path}'
    source_before = file_path.read_bytes()
    extracted = extract_markdown_objects(file_path)
    assert extracted, 'No Markdown objects were extracted'
    assert {'figure', 'table', 'list'} <= {obj['type'] for obj in extracted}
    client, summary_model = _configured_ollama_summary_client()
    print(f'Ollama model: {summary_model}')
    summaries = await summarize_markdown_object_tree_async(
        file_path, client, summary_model, lang='zh', show_progress=True)
    assert isinstance(summaries['blocks'], list)
    assert any(block.get('t') == 'Section' for block in summaries['blocks']), 'Missing section hierarchy'
    assert summaries['summary'].strip(), 'Missing document summary'
    nodes = list(_object_nodes(summaries))
    test_eq(len(nodes), len(extracted))
    for original, node in zip(extracted, nodes):
        summarized = node['object']
        test_eq(node['s'], summarized['summary'])
        test_eq({key: value for key, value in summarized.items() if key != 'summary'},
                {key: value for key, value in original.items() if key != 'summary'})
        assert isinstance(summarized['summary'], str) and summarized['summary'].strip(), (
            f"Missing summary for {summarized['type']} object {summarized['id']}")
    test_eq(file_path.read_bytes(), source_before)
    semantics_file = file_path.with_name(f'{file_path.stem}_semantics.json')
    _write_semantics_tree(semantics_file, summaries)
    test_eq(json.loads(semantics_file.read_text(encoding='utf-8')), summaries)
    print(f'Exported summaries: {semantics_file}')
    print(f'Summarized {len(nodes)} objects from {file_path.name}')
    return summaries



In [ ]:
# | notest
sx322018_object_summaries = await test_summarize_objects_with_sx322018_md()
sx322018_object_summaries
